# bge-m3 연구실 ↔ 교과목 매칭 실험 v3 (몇 개를, 어떻게 섞어 보여줄까)

v2 결론: 잡음 지표는 좋아졌지만 관련도는 그대로였다. 1~3위는 믿을 만하고 6위 이후는 대부분 잡음이며,
허브 보정·주제 분산을 강하게 걸면 관련도가 떨어지고 규칙 엔진 가산점이 이를 회복했다.

v3는 세 가지를 시험한다.

| 버전 | 허브 보정 λ | 주제 분산 | 규칙 가산 β | 설명 |
|---|---|---|---|---|
| A | – | – | – | v1 방식 (기준) |
| E | 0.7 | 강한 감점 | 0.04 | v2 최종 |
| F | 0.3 | 주제당 최소 1개 보장 | 0.04 | 보정·분산 완화 |
| G | 0.3 | 주제당 최소 1개 보장 | 0.08 | F + 규칙 비중 2배 |
| H | 0 | 없음 | 0.08 | 임베딩 + 규칙만 (단순 하이브리드) |

그리고 모든 버전에 **컷오프**(1위와 점수 차가 크면 보여주지 않음, 최소 3개·최대 10개)를 적용해
"10개 고정"과 "기준 이상만" 중 무엇이 나은지 비교한다.

### 준비
- **런타임 → 런타임 유형 변경 → T4 GPU**
- 업로드 (서버 저장소 `data/`): `courses.jsonl`, `lab_profiles.jsonl`, `course_profiles.jsonl`, `concepts.json`
- 업로드 (서버 `experiments/bge_m3/judge/`, 선택): `judge_key.json`, `judge_scores.json` — v2 채점을 재사용해 **새로 나온 과목만** 채점한다

In [ ]:
# 1. GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('⚠ GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 선택하고 다시 실행하세요.')

In [ ]:
# 2. 라이브러리 설치
!pip install -q sentence-transformers

In [ ]:
# 3. 데이터 업로드
import os
from google.colab import files
need = ['courses.jsonl', 'lab_profiles.jsonl', 'course_profiles.jsonl', 'concepts.json']
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print('업로드할 파일:', missing)
    files.upload()
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'아직 없는 파일: {missing}'
print('데이터 준비 완료')
if os.path.exists('judge_key.json') and os.path.exists('judge_scores.json'):
    os.rename('judge_key.json', 'v2_judge_key.json'); os.rename('judge_scores.json', 'v2_judge_scores.json')
    print('v2 채점 파일 확인 → 재사용합니다')
else:
    print('v2 채점 파일 없음 → 모든 쌍을 새로 채점합니다')

In [ ]:
# 4. 모델과 데이터 불러오기
import json, re, time, random, statistics
from collections import defaultdict, Counter
import numpy as np
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
t0 = time.time()
model = SentenceTransformer('BAAI/bge-m3', device=device)
if device == 'cuda':
    model.half()
model.max_seq_length = 256
print(f'모델 로드 {time.time()-t0:.1f}s, device={device}')

def enc(texts, bs=64):
    return model.encode(texts, batch_size=bs, normalize_embeddings=True, show_progress_bar=len(texts) > 200,
                        convert_to_numpy=True).astype(np.float32)

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

courses = load_jsonl('courses.jsonl')
course_by_id = {c['record_id']: c for c in courses}
profiles = [p for p in load_jsonl('lab_profiles.jsonl') if p.get('status') != 'insufficient']
course_profiles = {c['record_id']: c for c in load_jsonl('course_profiles.jsonl')}
concepts = {c['id']: c for c in json.load(open('concepts.json', encoding='utf-8'))['concepts']}
print(f'과목 {len(courses)}개, 추천 대상 연구실 {len(profiles)}곳')

## 1단계: 과목 조각 만들기 (v1 방식 / 정리한 방식)

- **v1 조각**: 설명, 학습목표, 강의계획 각 줄(15자 초과)
- **정리한 조각**: 30자 미만·의미 없는 줄 제거("See attached file", "Lecture Schedule", "TBD" 등), 강의계획은 3줄씩 묶어 문맥 유지

In [ ]:
def course_name(c):
    return c.get('course_name_kr') or c.get('course_name_en') or c['course_code']

def plan_lines(c):
    return [l.strip() for l in re.split(r'\n+', str(c.get('course_plan') or '')) if l.strip()]

# v1 조각 (버전 A)
def chunks_v1():
    out = []
    for c in courses:
        parts = [('설명', c.get('description') or ''), ('학습목표', c.get('objectives') or '')]
        parts += [(f'강의계획 {i+1}', p) for i, p in enumerate([p for p in plan_lines(c) if len(p) > 15][:20])]
        for where, text in parts:
            text = text.strip()
            if len(text) <= 15 or re.match(r'^\s*(선수과목|prerequisite)', text, re.I):
                continue
            out.append({'record_id': c['record_id'], 'where': where, 'text': text[:500]})
    return out

JUNK = re.compile(r'(see attached|attached file|첨부|lecture schedule|course schedule|tbd|tba|to be announced|midterm|final exam|중간고사|기말고사|^week\s*\d+\s*:?$|^\d+\s*주차\s*:?$)', re.I)
def meaningful(line):
    body = re.sub(r'^[\W\d_]*(week|주차|lecture|chapter|ch\.)?\s*[\d.]*\s*[:.)-]?\s*', '', line, flags=re.I)
    return len(body) >= 8 and not JUNK.search(line)

# 정리한 조각 (버전 B~E)
MIN_CHARS, PLAN_WINDOW = 30, 3
def chunks_clean():
    out = []
    for c in courses:
        for where, text in [('설명', c.get('description') or ''), ('학습목표', c.get('objectives') or '')]:
            text = text.strip()
            if len(text) >= MIN_CHARS and not re.match(r'^\s*(선수과목|prerequisite)', text, re.I):
                out.append({'record_id': c['record_id'], 'where': where, 'text': text[:500]})
        lines = [l for l in plan_lines(c) if meaningful(l)]
        for i in range(0, len(lines), PLAN_WINDOW):
            text = ' / '.join(lines[i:i+PLAN_WINDOW])
            if len(text) >= MIN_CHARS:
                out.append({'record_id': c['record_id'], 'where': f'강의계획 {i+1}~{min(i+PLAN_WINDOW, len(lines))}', 'text': text[:500]})
    return out

def encode_chunks(chunks):
    texts = [f"{course_name(course_by_id[m['record_id']])} | {m['text']}" for m in chunks]
    return enc(texts)

meta_A, meta_B = chunks_v1(), chunks_clean()
t0 = time.time()
emb_A, emb_B = encode_chunks(meta_A), encode_chunks(meta_B)
print(f'v1 조각 {len(meta_A)}개, 정리한 조각 {len(meta_B)}개, 벡터화 {time.time()-t0:.1f}s')
print('정리한 조각 예시:')
for m in random.Random(0).sample(meta_B, 3):
    print('  ', course_name(course_by_id[m['record_id']]), '|', m['where'], '|', m['text'][:90])

## 2단계: 연구실 주제 벡터화

In [ ]:
lab_topics = []          # (lab_id, topic)
for p in profiles:
    for t in p['topics']:
        if t.get('weight', 'primary') == 'primary':
            lab_topics.append((p['lab_id'], t))
emb_T = enc([f"{t['label_ko']} | {t['quote']}" for _, t in lab_topics])
topics_of = defaultdict(list)
for row, (lid, _) in enumerate(lab_topics):
    topics_of[lid].append(row)
print(f'연구실 주제 {len(lab_topics)}개')

## 3단계: 버전별 순위 계산

공통 구조: (연구실 주제 × 과목) 점수 → 연구실 × 과목 점수 → 상위 30개 목록과 근거 문장 쌍.
조절값(λ, γ, β)은 아래 셀 맨 위에서 바꿀 수 있습니다.

In [ ]:
RULE_BETA = 0.04     # rule_bonus의 기본 β (버전별로 덮어씀)
GENERIC = {'machine_learning', 'software_engineering', 'statistics_probability', 'optimization'}
TOP_LIST = 30

course_ids = [c['record_id'] for c in courses]
cidx = {rid: i for i, rid in enumerate(course_ids)}

def topic_course_scores(meta, emb, topk=2):
    """(주제 × 과목) 점수 = 그 과목 조각 중 상위 topk개 유사도 평균. 근거 조각 번호도 반환."""
    sim = emb_T @ emb.T                                   # topics x chunks
    owner = np.array([cidx[m['record_id']] for m in meta])
    S = np.full((len(lab_topics), len(courses)), -1.0, dtype=np.float32)
    best_chunk = np.full((len(lab_topics), len(courses)), -1, dtype=np.int64)
    for ci in range(len(courses)):
        cols = np.where(owner == ci)[0]
        if len(cols) == 0:
            continue
        block = sim[:, cols]
        k = min(topk, len(cols))
        part = -np.sort(-block, axis=1)[:, :k]
        S[:, ci] = part.mean(axis=1)
        best_chunk[:, ci] = cols[np.argmax(block, axis=1)]
    return S, best_chunk

def norm_name(name):
    return re.sub(r'[\s\-·]+', '', name).lower()

def rule_bonus(lab_id, rid, beta=None):
    beta = RULE_BETA if beta is None else beta
    prof = next(p for p in profiles if p['lab_id'] == lab_id)
    core = {t['concept'] for t in prof['topics'] if t.get('weight', 'primary') == 'primary' and t.get('scope', 'core') == 'core'}
    shared = [x for x in course_profiles.get(rid, {}).get('concepts', []) if x['concept'] in core]
    if not shared:
        return 0.0, []
    specific = [x for x in shared if x['concept'] not in GENERIC]
    if not specific:
        return beta * 0.25, shared
    return beta * (1.0 if any(x['strength'] == 'title' for x in specific) else 0.5), shared

def build_entry(lab_id, ci, score, row, chunk_meta, chunk_i):
    c = courses[ci]; t = lab_topics[row][1]; m = chunk_meta[chunk_i]
    return {'record_id': c['record_id'], 'code': c['course_code'], 'name': course_name(c), 'score': round(float(score), 4),
            'lab_topic': t['label_ko'], 'lab_quote': t['quote'][:160], 'course_where': m['where'], 'course_text': m['text'][:160]}

# ---- 버전 A: v1 방식 그대로 (조각 쌍 상위 3개 평균 + 여러 주제 가산) ----
def rank_v1():
    sim = emb_T @ emb_A.T
    out = {}
    for lid, rows in topics_of.items():
        best = defaultdict(list)
        for row in rows:
            for k in np.argsort(-sim[row])[:80]:
                best[cidx[meta_A[k]['record_id']]].append((float(sim[row, k]), row, int(k)))
        scored = []
        for ci, hits in best.items():
            hits.sort(reverse=True)
            n_topics = len({h[1] for h in hits if h[0] > 0.5})
            s = sum(h[0] for h in hits[:3]) / 3 + 0.03 * max(0, n_topics - 1)
            scored.append(build_entry(lid, ci, s, hits[0][1], meta_A, hits[0][2]))
        out[lid] = sorted(scored, key=lambda x: -x['score'])[:TOP_LIST]
    return out

S_B, chunk_B = topic_course_scores(meta_B, emb_B)
background = S_B.mean(axis=0)            # 과목이 모든 연구실 주제와 평균적으로 얼마나 가까운지

MIN1_MARGIN = 0.08   # '주제당 최소 1개'를 보장할 때, 그 주제 최고 과목이 1위와 이만큼 이내일 때만 보장

def rank_variant(lam=0.0, diversify=None, gamma=0.03, beta=0.0, dedupe=True):
    # diversify: None / 'penalty'(v2 방식) / 'min1'(주제당 최소 1개 보장 후 점수순)
    S = S_B - lam * background[None, :]
    out = {}
    for lid, rows in topics_of.items():
        sub = S[rows]
        best_row = sub.argmax(axis=0)
        cand = []
        for ci in range(len(courses)):
            r = rows[best_row[ci]]
            s = float(sub[best_row[ci], ci])
            if beta:
                s += rule_bonus(lid, courses[ci]['record_id'], beta)[0]
            cand.append([s, ci, r])
        cand.sort(key=lambda x: -x[0])
        cand = cand[:200]
        if diversify == 'min1':
            top = cand[0][0]
            firsts = {}
            for c in cand:
                if c[2] not in firsts and c[0] >= top - MIN1_MARGIN:
                    firsts[c[2]] = c
            guaranteed = sorted(firsts.values(), key=lambda x: -x[0])
            order = guaranteed + [c for c in cand if c not in guaranteed]
        elif diversify == 'penalty':
            pool, used, order = cand[:], Counter(), []
            while pool:
                j = max(range(len(pool)), key=lambda i: pool[i][0] - gamma * used[pool[i][2]])
                c = pool.pop(j); used[c[2]] += 1; order.append(c)
        else:
            order = cand
        picked, seen = [], set()
        for s, ci, r in order:
            if dedupe:
                k = norm_name(course_name(courses[ci]))
                if k in seen:
                    continue
                seen.add(k)
            picked.append(build_entry(lid, ci, s, r, meta_B, chunk_B[r, ci]))
            if len(picked) >= TOP_LIST:
                break
        out[lid] = picked
    return out

t0 = time.time()
VARIANT_SETTINGS = {
    'E': dict(lam=0.7, diversify='penalty', gamma=0.03, beta=0.04),
    'F': dict(lam=0.3, diversify='min1', beta=0.04),
    'G': dict(lam=0.3, diversify='min1', beta=0.08),
    'H': dict(lam=0.0, diversify=None, beta=0.08),
}
variants = {'A': rank_v1(), **{v: rank_variant(**kw) for v, kw in VARIANT_SETTINGS.items()}}

# 컷오프: 1위 점수에서 (1위 - 30위) 폭의 CUT_ALPHA 비율 안쪽만 보여줌. 최소 CUT_MIN, 최대 CUT_MAX개
CUT_ALPHA, CUT_MIN, CUT_MAX = 0.35, 3, 10
def cutoff(lst):
    if not lst:
        return []
    top, low = lst[0]['score'], lst[min(len(lst), 30) - 1]['score']
    thr = top - CUT_ALPHA * (top - low)
    keep = [c for c in lst[:CUT_MAX] if c['score'] >= thr]
    return keep if len(keep) >= CUT_MIN else lst[:CUT_MIN]
cut_lists = {v: {lid: cutoff(r[lid]) for lid in r} for v, r in variants.items()}
for v in variants:
    n = [len(x) for x in cut_lists[v].values()]
    print(f'{v}: 컷오프 후 평균 {statistics.mean(n):.1f}개 (최소 {min(n)}, 최대 {max(n)})')
print(f'버전 {list(variants)} 계산 {time.time()-t0:.1f}s')

## 4단계: 자동 건강 지표 (정답 없이)

| 지표 | 뜻 | 좋은 방향 |
|---|---|---|
| 은정답 재현율 | 규칙 엔진에서 **과목 제목**에 연구실 핵심 주제가 들어간 과목이 상위 10개에 들어간 비율 (사람이 봐도 거의 확실한 연결) | ↑ (단, E~H는 규칙을 섞어서 유리함) |
| 최대 허브 | 한 과목이 상위 10개에 등장한 연구실 비율의 최댓값 | ↓ |
| 다양성 | 전체 연구실 상위 10개에 등장한 서로 다른 과목 수 | ↑ |
| 주제 커버리지 | 상위 10개가 다룬 연구실 주제 수 ÷ min(주제 수, 10) | ↑ |
| 한 주제 독점 | 상위 10개 중 7개 이상이 한 주제인 연구실 수 | ↓ |
| 짧은 근거 | 근거 과목 조각이 35자 미만인 비율 | ↓ |
| 중복 | 같은 이름 과목이 상위 10개에 두 번 이상 | ↓ |
| 4개 학과·수학 외 | CSED·EECE·CITE·SEMI·DISU·MATH 외 과목 비율 (참고용, 맞는 경우도 있음) | 참고 |

In [ ]:
TOPN = 10
CORE_PREFIX = {'CSED', 'EECE', 'CITE', 'SEMI', 'DISU', 'MATH'}

silver = {}
for p in profiles:
    lid = p['lab_id']
    core = {t['concept'] for t in p['topics'] if t.get('weight', 'primary') == 'primary' and t.get('scope', 'core') == 'core'} - GENERIC
    s = {rid for rid, cp in course_profiles.items() if any(x['concept'] in core and x['strength'] == 'title' for x in cp['concepts'])}
    if s:
        silver[lid] = s

def metrics(ranking):
    labs = list(ranking)
    tops = {lid: ranking[lid][:TOPN] for lid in labs}
    appear = Counter(c['record_id'] for lst in tops.values() for c in lst)
    rec = [len(silver[l] & {c['record_id'] for c in tops[l]}) / min(TOPN, len(silver[l])) for l in labs if l in silver]
    cov, hog = [], 0
    for lid in labs:
        n_topics = len(topics_of[lid])
        tc = Counter(c['lab_topic'] for c in tops[lid])
        cov.append(len(tc) / min(n_topics, TOPN))
        hog += tc.most_common(1)[0][1] >= 7 if tc else 0
    allc = [c for lst in tops.values() for c in lst]
    dups = sum(len(lst) - len({norm_name(c['name']) for c in lst}) for lst in tops.values())
    return {
        '은정답 재현율': round(statistics.mean(rec), 3),
        '최대 허브': f"{max(appear.values())}/{len(labs)}",
        '다양성(서로 다른 과목)': len(appear),
        '주제 커버리지': round(statistics.mean(cov), 3),
        '한 주제 독점(곳)': hog,
        '짧은 근거': round(sum(len(c['course_text']) < 35 for c in allc) / len(allc), 3),
        '중복': dups,
        '4개 학과·수학 외': round(sum(c['code'][:4] not in CORE_PREFIX for c in allc) / len(allc), 3),
    }

import pandas as pd
table = pd.DataFrame({v: metrics(r) for v, r in variants.items()})
print(f'은정답이 있는 연구실 {len(silver)}곳 / 전체 {len(profiles)}곳')
table

In [ ]:
# 버전별로 가장 많이 반복 등장한 과목 (허브 확인)
for v, r in variants.items():
    appear = Counter((c['code'], c['name']) for lst in r.values() for c in lst[:TOPN])
    print(f"[{v}] " + ', '.join(f"{n}곳 {k[1]}" for k, n in appear.most_common(5)))

In [ ]:
# 버전 간 상위 10개 겹침 (Jaccard 평균) — 어떤 수정이 결과를 크게 바꿨는지
names = list(variants)
jac = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        vals = []
        for lid in variants[a]:
            x = {c['record_id'] for c in variants[a][lid][:TOPN]}; y = {c['record_id'] for c in variants[b][lid][:TOPN]}
            vals.append(len(x & y) / len(x | y))
        jac.loc[a, b] = round(statistics.mean(vals), 2)
jac

## 5단계: Claude Code 채점용 파일 만들기 (블라인드)

- v2와 같은 연구실 20곳(SEED 7)에 대해 **모든 버전의 상위 10개를 합친 뒤, v2에서 이미 채점한 쌍을 빼고** 섞어서 `judge_packet.json`을 만듭니다.
- 채점용 파일에는 과목이 **어느 버전에서 나왔는지 들어 있지 않습니다**. 버전 대응표는 `judge_key.json`에 따로 저장합니다.
- 이 zip을 Claude Code에 주면 `judge_packet.json`만 보고 과목마다 0~2점을 매겨 `judge_scores.json`을 돌려줍니다.
  - **2** 연구실 주제에 직접 필요한 지식·방법
  - **1** 기초 지식이거나 부분적으로 관련
  - **0** 무관하거나 같은 단어를 다른 뜻으로 쓴 경우
- 받은 `judge_scores.json`을 7단계에서 업로드하면 버전별 점수가 계산됩니다.
- AI 채점도 정답은 아닙니다. 점수가 엇갈리는 몇 개는 눈으로 확인해 보세요.

In [ ]:
N_JUDGE_LABS = 20
SEED = 7
prof_by_id = {p['lab_id']: p for p in profiles}
judge_labs = random.Random(SEED).sample([p['lab_id'] for p in profiles], min(N_JUDGE_LABS, len(profiles)))

prior = {}   # v2 채점 재사용: (lab_id, record_id) -> (score, reason)
if os.path.exists('v2_judge_key.json'):
    k2 = json.load(open('v2_judge_key.json', encoding='utf-8')); s2 = json.load(open('v2_judge_scores.json', encoding='utf-8'))
    prior = {(cid.split('#')[0], rid): (int(s2[cid]['score']), s2[cid].get('reason', '')) for cid, rid in k2.items() if cid in s2}
print(f'재사용하는 v2 채점 {len(prior)}쌍')

packet, key = [], {}
for lid in judge_labs:
    p = prof_by_id[lid]
    rids = list(dict.fromkeys(c['record_id'] for r in variants.values() for c in r[lid][:TOPN]))
    rids = [rid for rid in rids if (lid, rid) not in prior]
    if not rids:
        continue
    random.Random(lid).shuffle(rids)
    items = []
    for i, rid in enumerate(rids):
        c = course_by_id[rid]
        items.append({'id': f'{lid}#N{i:02d}', 'code': c['course_code'], 'name_kr': c.get('course_name_kr') or '',
                      'name_en': c.get('course_name_en') or '',
                      'description': (c.get('description') or '')[:400], 'objectives': (c.get('objectives') or '')[:300]})
        key[f'{lid}#N{i:02d}'] = rid
    packet.append({'lab_id': lid, 'professors': p['professors'], 'summary_ko': p['summary_ko'],
                   'topics': [{'label': t['label_ko'], 'quote': t['quote'][:250]} for t in p['topics'] if t.get('weight', 'primary') == 'primary'],
                   'courses': items})
json.dump({'instructions': '각 course에 0~2점: 2=연구실 주제에 직접 필요한 지식·방법, 1=기초 또는 부분 관련, 0=무관/단어만 겹침. '
                           '반환 형식: {"<course id>": {"score": 0|1|2, "reason": "한 문장"}}',
           'labs': packet}, open('judge_packet.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
json.dump(key, open('judge_key.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(f'새로 채점할 쌍: 연구실 {len(packet)}곳, {len(key)}개 → judge_packet.json (v2와 겹치는 쌍은 제외)')

## 7단계 (채점 후): Claude Code 채점 결과 불러오기

Claude Code가 돌려준 `judge_scores.json`을 업로드하면 버전별 점수를 계산합니다. 아직 없으면 건너뛰세요.

In [ ]:
judged = dict(prior)   # v2 채점 + v3 새 채점
if not os.path.exists('judge_scores.json'):
    print('Claude Code가 돌려준 judge_scores.json이 없으면 업로드 창에서 취소하고 넘어가도 됩니다.')
    try:
        files.upload()
    except Exception:
        pass
if os.path.exists('judge_scores.json'):
    new = json.load(open('judge_scores.json', encoding='utf-8'))
    for cid, v in new.items():
        if cid in key:
            judged[(cid.split('#')[0], key[cid])] = (int(v['score']), v.get('reason', ''))
need = {(lid, c['record_id']) for r in variants.values() for lid in judge_labs for c in r[lid][:TOPN]}
print(f'채점된 쌍 {len(need & set(judged))} / 필요한 {len(need)}')

def summarize(lists):
    sc = [judged[(lid, c['record_id'])][0] for lid in judge_labs for c in lists[lid] if (lid, c['record_id']) in judged]
    good_all = {pair for pair in need if judged.get(pair, (0,))[0] >= 1}
    shown_good = {(lid, c['record_id']) for lid in judge_labs for c in lists[lid] if judged.get((lid, c['record_id']), (0,))[0] >= 1}
    return {'보여준 개수(평균)': round(statistics.mean(len(lists[l]) for l in judge_labs), 1),
            '평균 점수(0~2)': round(statistics.mean(sc), 3) if sc else None,
            '0점 비율': round(sum(x == 0 for x in sc) / len(sc), 3) if sc else None,
            '2점 비율': round(sum(x == 2 for x in sc) / len(sc), 3) if sc else None,
            '관련 과목 회수율': round(len(shown_good) / len(good_all), 3) if good_all else None}

if len(need & set(judged)) == len(need):
    rows = {}
    for v in variants:
        rows[f'{v} · 10개'] = summarize({l: variants[v][l][:TOPN] for l in judge_labs})
        rows[f'{v} · 컷오프'] = summarize({l: cut_lists[v][l] for l in judge_labs})
    display(pd.DataFrame(rows).T)
    print('관련 과목 회수율 = 모든 버전 후보 중 1점 이상인 과목을 이 목록이 얼마나 담았는지')
    print('\n순위별 평균 점수 (1~10위):')
    for v in variants:
        pos = [statistics.mean(judged[(l, variants[v][l][i]['record_id'])][0] for l in judge_labs) for i in range(TOPN)]
        print(f'  {v}: ' + ' '.join(f'{x:.2f}' for x in pos))
else:
    print('아직 채점되지 않은 쌍이 있습니다 — judge_packet.json을 Claude Code에 채점받은 뒤 이 셀을 다시 실행하세요.')

## 8단계: 나란히 보기 (A 10개 vs G 컷오프)

In [ ]:
def side_by_side(prof, n=10):
    p = next((x for x in profiles if any(prof in name for name in x['professors'])), None)
    if p is None:
        print(f'{prof}: 결과 없음'); return
    lid = p['lab_id']
    print(f"\n■ {' · '.join(p['professors'])} — {p['summary_ko']}")
    a, e = variants['A'][lid][:n], cut_lists['G'][lid]
    for i in range(n):
        left = f"{a[i]['code']} {a[i]['name']}"[:34] if i < len(a) else ''
        right = f"{e[i]['code']} {e[i]['name']}" if i < len(e) else ''
        mark = ''
        if i < len(e) and (lid, e[i]['record_id']) in judged:
            mark = f"  (AI {judged[(lid, e[i]['record_id'])][0]}점)"
        print(f"  {i+1:>2}. A: {left:<36} | G: {right}{mark}")
    print('  G의 근거 예시:')
    for c in e[:3]:
        print(f"     {c['name']}: [{c['lab_topic']}] ↔ ({c['course_where']}) {c['course_text'][:70]}")

for prof in ['한수희', '김광인', '서영주', '이근배']:
    side_by_side(prof)

## 결과 저장·다운로드

In [ ]:
import zipfile
summary = {v: {lid: {'top30': r[lid], 'cutoff_ids': [c['record_id'] for c in cut_lists[v][lid]]} for lid in r} for v, r in variants.items()}
json.dump(summary, open('rankings_v3.json', 'w', encoding='utf-8'), ensure_ascii=False)
table.to_csv('metrics_v3.csv', encoding='utf-8-sig')
json.dump({'VARIANTS': VARIANT_SETTINGS, 'MIN1_MARGIN': MIN1_MARGIN, 'CUT_ALPHA': CUT_ALPHA, 'CUT_MIN': CUT_MIN,
           'CUT_MAX': CUT_MAX, 'MIN_CHARS': MIN_CHARS, 'PLAN_WINDOW': PLAN_WINDOW, 'N_JUDGE_LABS': N_JUDGE_LABS, 'SEED': SEED},
          open('settings_v3.json', 'w'), indent=1)
with zipfile.ZipFile('bge_m3_results_v3.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['rankings_v3.json', 'metrics_v3.csv', 'settings_v3.json', 'judge_packet.json', 'judge_key.json']:
        z.write(f)
print('저장 완료: bge_m3_results_v3.zip')
files.download('bge_m3_results_v3.zip')